# Proyecto 1 — Predicción de popularidad de canción
## Versión 6 — Feature Engineering Avanzado + Pseudo-labeling + Stacking

### Mejoras vs v5
| Mejora | Detalle |
|---|---|
| **Más features de artista** | media, mediana, hit_rate (% > 60), nro de géneros, explícitas ratio |
| **Features de texto** | track_name: has_feat, has_remix, length; album: tracks_per_album |
| **Target encoding con suavizado Bayesiano** | Evita leakage y reduce varianza en artistas raros |
| **Interacciones cúbicas** | Más features de audio cruzadas |
| **Pseudo-labeling (1 ronda)** | Usa predicciones confiables del test para ampliar train |
| **HistGradientBoosting** | Cuarto modelo base — más diversidad en el ensemble |
| **Meta-modelo XGBoost** | Reemplaza Ridge por un meta-modelo no-lineal |


## 1. Instalaciones y librerías

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'optuna', 'xgboost', 'catboost', '--quiet'], capture_output=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import re
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
N_FOLDS = 5
print('✅ Librerías listas')

## 2. Carga de datos

In [ ]:
dataTraining = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTrain_Spotify.csv')
dataTesting  = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTest_Spotify.csv', index_col=0)

dataTraining = dataTraining.drop('Unnamed: 0', axis=1)
print(f'Train: {dataTraining.shape} | Test: {dataTesting.shape}')
dataTraining.head(2)

## 3. Feature Engineering Avanzado — sin leakage

### Novedades vs v5:
- **Target encoding con suavizado** (Bayesian smoothing): artista y género
- **Features de texto**: `track_name` y `album_name` → has_feat, has_remix, etc.
- **Más stats de artista**: mean, median, hit_rate, explicit_ratio, genre_diversity
- **Features de álbum**: tracks_per_album
- **Más interacciones de audio**

In [ ]:
def smooth_target_encode(train_df, val_df, test_df, group_col, target_col, global_mean, k=10):
    """
    Target encoding con suavizado Bayesiano.
    Para grupos pequeños (count < k) el encoding se acerca al global_mean.
    Esto reduce el overfitting en artistas/géneros raros.
    """
    agg = train_df.groupby(group_col)[target_col].agg(['mean', 'count'])
    agg['smooth'] = (agg['count'] * agg['mean'] + k * global_mean) / (agg['count'] + k)
    smooth_map = agg['smooth'].to_dict()
    col_name = f'{group_col}_te'
    for df in [val_df, test_df]:
        df[col_name] = df[group_col].map(smooth_map).fillna(global_mean)
    # Para train usamos Leave-One-Out para evitar leakage
    loo = train_df[target_col].sum() - train_df[target_col]
    loo_count = len(train_df) - 1
    train_df[col_name] = loo / loo_count  # aproximación global LOO
    # Alternativa más precisa por grupo:
    group_sum   = train_df.groupby(group_col)[target_col].transform('sum')
    group_count = train_df.groupby(group_col)[target_col].transform('count')
    loo_group = (group_sum - train_df[target_col]) / (group_count - 1).clip(lower=1)
    # Suavizado también para LOO
    smooth_loo = (group_count * loo_group + k * global_mean) / (group_count + k)
    train_df[col_name] = smooth_loo
    return train_df, val_df, test_df


def extract_text_features(df):
    """Extrae features útiles de campos de texto sin usar el target."""
    # Track name features
    name = df['track_name'].fillna('').str.lower()
    df['has_feat']    = name.str.contains(r'feat\.|ft\.| x ', regex=True).astype(int)
    df['has_remix']   = name.str.contains('remix|edit|version|remaster', regex=True).astype(int)
    df['has_live']    = name.str.contains('live|en vivo|ao vivo', regex=True).astype(int)
    df['track_name_len'] = name.str.len()
    df['track_words'] = name.str.split().str.len().fillna(0)
    df['has_paren']   = name.str.contains(r'\(|\[', regex=True).astype(int)
    
    # Artist features
    artists = df['artists'].fillna('')
    df['n_artists']       = artists.str.count(';') + 1  # artistas separados por ';'
    df['artist_name_len'] = artists.str.len()
    
    # Album features
    album = df['album_name'].fillna('').str.lower()
    df['album_is_single'] = album.str.contains('single|- single', regex=True).astype(int)
    df['album_name_len']  = album.str.len()
    return df


def build_features_fold(df_train_fold, df_val_fold, df_test):
    """
    Feature engineering sin leakage — versión avanzada.
    Todas las agregaciones sobre el target se calculan SOLO sobre df_train_fold.
    """
    train = df_train_fold.copy()
    val   = df_val_fold.copy()
    test  = df_test.copy()

    # ── Explicit → numérico ──────────────────────────────────────────────
    for df in [train, val, test]:
        df['explicit'] = df['explicit'].astype(int)

    global_mean   = train['popularity'].mean()
    global_median = train['popularity'].median()
    global_p75    = train['popularity'].quantile(0.75)

    # ── Features de texto (sin target) ──────────────────────────────────
    for df in [train, val, test]:
        extract_text_features(df)

    # ── Aggregation features por ARTISTA ────────────────────────────────
    artist_agg = (
        train.groupby('artists')['popularity']
        .agg(
            artist_mean  = 'mean',
            artist_median= 'median',
            artist_std   = 'std',
            artist_max   = 'max',
            artist_count = 'count',
            artist_p75   = lambda x: x.quantile(0.75),
            artist_min   = 'min',
            # Hit rate: % canciones con popularidad > 60
            artist_hit_rate = lambda x: (x > 60).mean(),
        )
        .reset_index()
    )
    artist_agg['artist_std']   = artist_agg['artist_std'].fillna(0)
    artist_agg['artist_range'] = artist_agg['artist_max'] - artist_agg['artist_min']
    artist_agg['artist_cv']    = artist_agg['artist_std'] / (artist_agg['artist_mean'] + 1e-6)
    artist_agg = artist_agg.drop(columns=['artist_min'])

    # Ratio de canciones explícitas por artista
    artist_explicit = train.groupby('artists')['explicit'].mean().reset_index()
    artist_explicit.columns = ['artists', 'artist_explicit_ratio']
    artist_agg = artist_agg.merge(artist_explicit, on='artists', how='left')

    # Diversidad de géneros del artista
    artist_genre_div = train.groupby('artists')['track_genre'].nunique().reset_index()
    artist_genre_div.columns = ['artists', 'artist_genre_diversity']
    artist_agg = artist_agg.merge(artist_genre_div, on='artists', how='left')

    ARTIST_COLS = ['artist_mean','artist_median','artist_std','artist_max',
                   'artist_count','artist_p75','artist_range','artist_cv',
                   'artist_hit_rate','artist_explicit_ratio','artist_genre_diversity']

    for df in [train, val, test]:
        for col in ARTIST_COLS:
            if col in df.columns:
                df.drop(columns=[col], inplace=True)
        merged = df.merge(artist_agg, on='artists', how='left')
        for col in ARTIST_COLS:
            df[col] = merged[col].values
        # Fillnas
        df['artist_mean']   = df['artist_mean'].fillna(global_mean)
        df['artist_median'] = df['artist_median'].fillna(global_median)
        df['artist_max']    = df['artist_max'].fillna(global_mean)
        df['artist_p75']    = df['artist_p75'].fillna(global_p75)
        df['artist_range']  = df['artist_range'].fillna(0)
        df['artist_std']    = df['artist_std'].fillna(0)
        df['artist_cv']     = df['artist_cv'].fillna(0)
        df['artist_count']  = df['artist_count'].fillna(1)
        df['artist_hit_rate']        = df['artist_hit_rate'].fillna(0)
        df['artist_explicit_ratio']  = df['artist_explicit_ratio'].fillna(0)
        df['artist_genre_diversity'] = df['artist_genre_diversity'].fillna(1)

    # ── Aggregation features por GÉNERO ─────────────────────────────────
    genre_agg = (
        train.groupby('track_genre')['popularity']
        .agg(genre_mean='mean', genre_std='std', genre_max='max',
             genre_median='median', genre_count='count')
        .reset_index()
    )
    genre_agg['genre_std'] = genre_agg['genre_std'].fillna(0)

    for df in [train, val, test]:
        for col in ['genre_mean','genre_std','genre_max','genre_median','genre_count']:
            if col in df.columns:
                df.drop(columns=[col], inplace=True)
        merged = df.merge(genre_agg, on='track_genre', how='left')
        for col in ['genre_mean','genre_std','genre_max','genre_median','genre_count']:
            df[col] = merged[col].fillna(global_mean).values

    # ── Target encoding con suavizado (artista y género) ─────────────────
    train, val, test = smooth_target_encode(train, val, test, 'artists', 'popularity', global_mean, k=10)
    train, val, test = smooth_target_encode(train, val, test, 'track_genre', 'popularity', global_mean, k=5)

    # ── Aggregations por ÁLBUM ──────────────────────────────────────────
    album_agg = (
        train.groupby('album_name')['popularity']
        .agg(album_mean='mean', album_count='count')
        .reset_index()
    )
    for df in [train, val, test]:
        for col in ['album_mean', 'album_count']:
            if col in df.columns:
                df.drop(columns=[col], inplace=True)
        merged = df.merge(album_agg, on='album_name', how='left')
        df['album_mean']  = merged['album_mean'].fillna(global_mean).values
        df['album_count'] = merged['album_count'].fillna(1).values

    # ── Features de audio — interacciones ──────────────────────────────
    for df in [train, val, test]:
        # v5 features
        df['loudness_energy']   = df['loudness'] * df['energy']
        df['dance_valence']     = df['danceability'] * df['valence']
        df['speech_instrument'] = df['speechiness'] + df['instrumentalness']
        df['liveness_explicit'] = df['liveness'] * df['explicit']
        # Nuevas interacciones
        df['energy_valence']    = df['energy'] * df['valence']
        df['dance_energy']      = df['danceability'] * df['energy']
        df['acoustic_instrument']= df['acousticness'] * df['instrumentalness']
        df['duration_min']      = df['duration_ms'] / 60000
        df['is_short_track']    = (df['duration_ms'] < 120000).astype(int)
        df['is_long_track']     = (df['duration_ms'] > 300000).astype(int)
        # Normalización de loudness (suele ser negativo, ej: -60 a 0)
        df['loudness_norm']     = (df['loudness'] + 60) / 60
        # Clave: modo × tonalidad como combinación
        df['key_mode']          = df['key'] * 2 + df['mode']
        # Artist popularity relativa al género
        df['artist_vs_genre']   = df['artist_mean'] - df['genre_mean']
        # Popularidad relativa del artista (artist_mean / global)
        df['artist_rel_pop']    = df['artist_mean'] / (global_mean + 1e-6)

    # ── Drop columnas de texto/ID ────────────────────────────────────────
    DROP_COLS = ['track_id', 'track_name', 'album_name', 'artists', 'track_genre']
    X_tr   = train.drop(columns=['popularity'] + DROP_COLS)
    y_tr   = train['popularity']
    X_vl   = val.drop(columns=['popularity'] + DROP_COLS)
    y_vl   = val['popularity']
    X_test = test.drop(columns=[c for c in DROP_COLS if c in test.columns])

    return X_tr, y_tr, X_vl, y_vl, X_test


print('✅ build_features_fold avanzado listo')
print(f'   Probando con una muestra del dataset...')
# Test rápido
_idx = np.arange(len(dataTraining))
_tr, _vl = train_test_split(_idx, test_size=0.1, random_state=0)
_Xtr, _ytr, _Xvl, _yvl, _Xte = build_features_fold(
    dataTraining.iloc[_tr].reset_index(drop=True),
    dataTraining.iloc[_vl].reset_index(drop=True),
    dataTesting
)
print(f'   Features generadas: {_Xtr.shape[1]}')
print(f'   Columnas: {list(_Xtr.columns)}')

## 4. Optimización de hiperparámetros con Optuna

Más trials y rango de búsqueda más amplio vs v5.

In [ ]:
idx_all = np.arange(len(dataTraining))
idx_tr, idx_vl = train_test_split(idx_all, test_size=0.20, random_state=RANDOM_STATE)

df_opt_tr = dataTraining.iloc[idx_tr].reset_index(drop=True)
df_opt_vl = dataTraining.iloc[idx_vl].reset_index(drop=True)

X_opt_tr, y_opt_tr, X_opt_vl, y_opt_vl, _ = build_features_fold(
    df_opt_tr, df_opt_vl, dataTesting
)
print(f'Optuna — train: {X_opt_tr.shape} | val: {X_opt_vl.shape}')

In [ ]:
# ── LIGHTGBM ─────────────────────────────────────────────────────────────────
def objective_lgbm(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 500, 3000),
        'max_depth':         trial.suggest_int('max_depth', 3, 14),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 512),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 150),
        'subsample':         trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 20.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 20.0, log=True),
        'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 1.0),
        'verbose': -1, 'random_state': RANDOM_STATE, 'n_jobs': -1,
    }
    m = lgb.LGBMRegressor(**params)
    m.fit(X_opt_tr, y_opt_tr,
          eval_set=[(X_opt_vl, y_opt_vl)],
          callbacks=[lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(period=-1)])
    return np.sqrt(mean_squared_error(y_opt_vl, np.clip(m.predict(X_opt_vl), 0, 100)))

study_lgbm = optuna.create_study(direction='minimize')
study_lgbm.optimize(objective_lgbm, n_trials=75, show_progress_bar=True)
print(f'\n✅ LGBM — Mejor RMSE val: {study_lgbm.best_value:.4f}')

In [ ]:
# ── XGBOOST ──────────────────────────────────────────────────────────────────
def objective_xgb(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 500, 3000),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'subsample':         trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 20.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 20.0, log=True),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 30),
        'gamma':             trial.suggest_float('gamma', 0.0, 5.0),
        'max_delta_step':    trial.suggest_int('max_delta_step', 0, 10),
        'tree_method': 'hist', 'random_state': RANDOM_STATE,
        'n_jobs': -1, 'verbosity': 0,
    }
    m = xgb.XGBRegressor(**params)
    m.fit(X_opt_tr, y_opt_tr,
          eval_set=[(X_opt_vl, y_opt_vl)],
          early_stopping_rounds=50, verbose=False)
    return np.sqrt(mean_squared_error(y_opt_vl, np.clip(m.predict(X_opt_vl), 0, 100)))

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=75, show_progress_bar=True)
print(f'\n✅ XGB — Mejor RMSE val: {study_xgb.best_value:.4f}')

In [ ]:
# ── CATBOOST ─────────────────────────────────────────────────────────────────
def objective_cat(trial):
    params = {
        'iterations':          trial.suggest_int('iterations', 500, 3000),
        'depth':               trial.suggest_int('depth', 4, 10),
        'learning_rate':       trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 1e-3, 20.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 2.0),
        'random_strength':     trial.suggest_float('random_strength', 0.0, 2.0),
        'border_count':        trial.suggest_int('border_count', 32, 255),
        'min_data_in_leaf':    trial.suggest_int('min_data_in_leaf', 1, 50),
        'random_seed': RANDOM_STATE, 'verbose': 0,
    }
    m = CatBoostRegressor(**params)
    m.fit(X_opt_tr, y_opt_tr,
          eval_set=(X_opt_vl, y_opt_vl),
          early_stopping_rounds=50, verbose=False)
    return np.sqrt(mean_squared_error(y_opt_vl, np.clip(m.predict(X_opt_vl), 0, 100)))

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(objective_cat, n_trials=75, show_progress_bar=True)
print(f'\n✅ CatBoost — Mejor RMSE val: {study_cat.best_value:.4f}')

In [ ]:
# ── HistGradientBoosting (nuevo modelo base) ──────────────────────────────────
def objective_hgb(trial):
    params = {
        'max_iter':           trial.suggest_int('max_iter', 200, 1000),
        'max_depth':          trial.suggest_int('max_depth', 3, 12),
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_leaf_nodes':     trial.suggest_int('max_leaf_nodes', 10, 255),
        'min_samples_leaf':   trial.suggest_int('min_samples_leaf', 5, 100),
        'l2_regularization':  trial.suggest_float('l2_regularization', 1e-6, 10.0, log=True),
        'random_state': RANDOM_STATE,
    }
    m = HistGradientBoostingRegressor(**params)
    m.fit(X_opt_tr, y_opt_tr)
    return np.sqrt(mean_squared_error(y_opt_vl, np.clip(m.predict(X_opt_vl), 0, 100)))

study_hgb = optuna.create_study(direction='minimize')
study_hgb.optimize(objective_hgb, n_trials=50, show_progress_bar=True)
print(f'\n✅ HGB — Mejor RMSE val: {study_hgb.best_value:.4f}')

print('\n📊 Comparación modelos base (Optuna):')
for name, study in [('LightGBM', study_lgbm), ('XGBoost', study_xgb),
                     ('CatBoost', study_cat), ('HistGBM', study_hgb)]:
    print(f'  {name:<12}: {study.best_value:.4f}')

## 5. Stacking KFold con 4 modelos base — sin leakage

In [ ]:
params_lgbm = study_lgbm.best_params.copy()
params_lgbm.update({'verbose': -1, 'random_state': RANDOM_STATE, 'n_jobs': -1})

params_xgb = study_xgb.best_params.copy()
params_xgb.update({'tree_method': 'hist', 'random_state': RANDOM_STATE,
                    'n_jobs': -1, 'verbosity': 0})

params_cat = study_cat.best_params.copy()
params_cat.update({'random_seed': RANDOM_STATE, 'verbose': 0})

params_hgb = study_hgb.best_params.copy()
params_hgb.update({'random_state': RANDOM_STATE})

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
all_indices = np.arange(len(dataTraining))
y_full = dataTraining['popularity'].values

# OOF arrays
oof_lgbm = np.zeros(len(dataTraining))
oof_xgb  = np.zeros(len(dataTraining))
oof_cat  = np.zeros(len(dataTraining))
oof_hgb  = np.zeros(len(dataTraining))

# Test prediction arrays
test_lgbm = np.zeros(len(dataTesting))
test_xgb  = np.zeros(len(dataTesting))
test_cat  = np.zeros(len(dataTesting))
test_hgb  = np.zeros(len(dataTesting))

rmse_lgbm_folds = []
rmse_xgb_folds  = []
rmse_cat_folds  = []
rmse_hgb_folds  = []

print('Entrenando stacking con 4 modelos...\n')

for fold, (train_idx, val_idx) in enumerate(kf.split(all_indices)):
    print(f'─── Fold {fold+1}/{N_FOLDS} ──────────────────────────')

    df_tr_raw = dataTraining.iloc[train_idx].reset_index(drop=True)
    df_vl_raw = dataTraining.iloc[val_idx].reset_index(drop=True)

    X_tr, y_tr, X_vl, y_vl, X_test = build_features_fold(
        df_tr_raw, df_vl_raw, dataTesting
    )

    # LightGBM
    m_lgbm = lgb.LGBMRegressor(**params_lgbm)
    m_lgbm.fit(X_tr, y_tr,
               eval_set=[(X_vl, y_vl)],
               callbacks=[lgb.early_stopping(100, verbose=False),
                           lgb.log_evaluation(period=-1)])
    oof_lgbm[val_idx]  = np.clip(m_lgbm.predict(X_vl), 0, 100)
    test_lgbm         += np.clip(m_lgbm.predict(X_test), 0, 100) / N_FOLDS
    r_l = np.sqrt(mean_squared_error(y_vl, oof_lgbm[val_idx]))
    rmse_lgbm_folds.append(r_l)
    print(f'  LGBM  RMSE = {r_l:.4f}')

    # XGBoost
    m_xgb = xgb.XGBRegressor(**params_xgb)
    m_xgb.fit(X_tr, y_tr,
              eval_set=[(X_vl, y_vl)],
              early_stopping_rounds=100, verbose=False)
    oof_xgb[val_idx]  = np.clip(m_xgb.predict(X_vl), 0, 100)
    test_xgb         += np.clip(m_xgb.predict(X_test), 0, 100) / N_FOLDS
    r_x = np.sqrt(mean_squared_error(y_vl, oof_xgb[val_idx]))
    rmse_xgb_folds.append(r_x)
    print(f'  XGB   RMSE = {r_x:.4f}')

    # CatBoost
    m_cat = CatBoostRegressor(**params_cat)
    m_cat.fit(X_tr, y_tr,
              eval_set=(X_vl, y_vl),
              early_stopping_rounds=100, verbose=False)
    oof_cat[val_idx]  = np.clip(m_cat.predict(X_vl), 0, 100)
    test_cat         += np.clip(m_cat.predict(X_test), 0, 100) / N_FOLDS
    r_c = np.sqrt(mean_squared_error(y_vl, oof_cat[val_idx]))
    rmse_cat_folds.append(r_c)
    print(f'  CAT   RMSE = {r_c:.4f}')

    # HistGradientBoosting
    m_hgb = HistGradientBoostingRegressor(**params_hgb)
    m_hgb.fit(X_tr, y_tr)
    oof_hgb[val_idx]  = np.clip(m_hgb.predict(X_vl), 0, 100)
    test_hgb         += np.clip(m_hgb.predict(X_test), 0, 100) / N_FOLDS
    r_h = np.sqrt(mean_squared_error(y_vl, oof_hgb[val_idx]))
    rmse_hgb_folds.append(r_h)
    print(f'  HGB   RMSE = {r_h:.4f}')
    print()

print('✅ Stacking completo')

In [ ]:
rmse_lgbm_oof = np.sqrt(mean_squared_error(y_full, oof_lgbm))
rmse_xgb_oof  = np.sqrt(mean_squared_error(y_full, oof_xgb))
rmse_cat_oof  = np.sqrt(mean_squared_error(y_full, oof_cat))
rmse_hgb_oof  = np.sqrt(mean_squared_error(y_full, oof_hgb))

print('📊 RMSE OOF por modelo:')
print(f'  LightGBM  : {rmse_lgbm_oof:.4f}  ± {np.std(rmse_lgbm_folds):.4f}')
print(f'  XGBoost   : {rmse_xgb_oof:.4f}  ± {np.std(rmse_xgb_folds):.4f}')
print(f'  CatBoost  : {rmse_cat_oof:.4f}  ± {np.std(rmse_cat_folds):.4f}')
print(f'  HistGBM   : {rmse_hgb_oof:.4f}  ± {np.std(rmse_hgb_folds):.4f}')

## 6. Ensembles y meta-modelo

In [ ]:
# ─── ENSEMBLE 1: Promedio simple ─────────────────────────────────────────────
oof_simple  = (oof_lgbm + oof_xgb + oof_cat + oof_hgb) / 4
test_simple = (test_lgbm + test_xgb + test_cat + test_hgb) / 4
rmse_simple = np.sqrt(mean_squared_error(y_full, oof_simple))
print(f'🔀 Promedio simple         RMSE OOF: {rmse_simple:.4f}')

# ─── ENSEMBLE 2: Ponderado por RMSE-inverso ───────────────────────────────────
ws = [1/rmse_lgbm_oof, 1/rmse_xgb_oof, 1/rmse_cat_oof, 1/rmse_hgb_oof]
wt = sum(ws)
oof_weighted  = sum(w * o for w, o in zip(ws, [oof_lgbm, oof_xgb, oof_cat, oof_hgb])) / wt
test_weighted = sum(w * t for w, t in zip(ws, [test_lgbm, test_xgb, test_cat, test_hgb])) / wt
rmse_weighted = np.sqrt(mean_squared_error(y_full, oof_weighted))
print(f'⚖️  Ponderado RMSE-inverso  RMSE OOF: {rmse_weighted:.4f}')

# ─── ENSEMBLE 3: Meta-modelo XGBoost (más expresivo que Ridge) ────────────────
meta_X_train = np.column_stack([oof_lgbm, oof_xgb, oof_cat, oof_hgb])
meta_X_test  = np.column_stack([test_lgbm, test_xgb, test_cat, test_hgb])

# Búsqueda del mejor meta-XGB con CV
best_meta_rmse = np.inf
best_meta_params = None
kf_meta = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for lr in [0.05, 0.1, 0.3]:
    for depth in [2, 3, 4]:
        for n_est in [100, 300]:
            rmse_tmp = []
            for tr_i, vl_i in kf_meta.split(meta_X_train):
                m = xgb.XGBRegressor(n_estimators=n_est, max_depth=depth,
                                     learning_rate=lr, random_state=RANDOM_STATE,
                                     n_jobs=-1, verbosity=0)
                m.fit(meta_X_train[tr_i], y_full[tr_i])
                p = np.clip(m.predict(meta_X_train[vl_i]), 0, 100)
                rmse_tmp.append(np.sqrt(mean_squared_error(y_full[vl_i], p)))
            avg = np.mean(rmse_tmp)
            if avg < best_meta_rmse:
                best_meta_rmse = avg
                best_meta_params = {'n_estimators': n_est, 'max_depth': depth, 'learning_rate': lr}

meta_xgb = xgb.XGBRegressor(**best_meta_params, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
meta_xgb.fit(meta_X_train, y_full)
oof_stacked  = np.clip(meta_xgb.predict(meta_X_train), 0, 100)
test_stacked = np.clip(meta_xgb.predict(meta_X_test), 0, 100)
rmse_stacked = np.sqrt(mean_squared_error(y_full, oof_stacked))
print(f'🧠 Meta XGBoost            RMSE OOF: {rmse_stacked:.4f}  params={best_meta_params}')

# Selección del mejor
scores = {
    'Promedio simple':    (rmse_simple,   test_simple),
    'Ponderado':          (rmse_weighted, test_weighted),
    'Meta XGBoost':       (rmse_stacked,  test_stacked),
}
best_name = min(scores, key=lambda k: scores[k][0])
best_rmse, best_preds = scores[best_name]

print('\n🏆 Ranking:')
for name, (rmse, _) in sorted(scores.items(), key=lambda x: x[1][0]):
    marker = ' ← MEJOR' if name == best_name else ''
    print(f'  {name:<22} RMSE OOF: {rmse:.4f}{marker}')

## 7. Pseudo-labeling 🔁

Usamos las predicciones más **confiables** del test (aquellas donde los 4 modelos coinciden, i.e., baja varianza entre ellos) como datos de entrenamiento adicionales. Luego reentrenamos LGBM (el más rápido) sobre el conjunto ampliado.

**Regla de confianza**: solo incluimos puntos donde la desviación estándar entre los 4 modelos < umbral (ej: 5 puntos).

In [ ]:
# ─── Pseudo-labeling ─────────────────────────────────────────────────────────
PSEUDO_STD_THRESHOLD = 5.0   # solo usamos puntos donde los 4 modelos están de acuerdo
PSEUDO_WEIGHT        = 0.5   # peso de las pseudo-labels vs labels reales

# Stack de predicciones test de los 4 modelos
test_preds_stack = np.column_stack([test_lgbm, test_xgb, test_cat, test_hgb])
test_std = test_preds_stack.std(axis=1)
confident_mask = test_std < PSEUDO_STD_THRESHOLD

print(f'Puntos test confiables (std < {PSEUDO_STD_THRESHOLD}): {confident_mask.sum()} / {len(confident_mask)} ({confident_mask.mean()*100:.1f}%)')

# Pseudo-labels = promedio de los 4 modelos para puntos confiables
pseudo_labels = best_preds[confident_mask]
dataTesting_confident = dataTesting[confident_mask].copy()
dataTesting_confident['popularity'] = pseudo_labels

# Dataset ampliado: train real + pseudo-labels
dataAugmented = pd.concat([dataTraining, dataTesting_confident], ignore_index=True)
print(f'Dataset ampliado: {len(dataTraining)} + {confident_mask.sum()} = {len(dataAugmented)} filas')

In [ ]:
# Reentrenamos LGBM (más rápido) sobre el dataset ampliado usando todo el train original como val
X_aug_full, y_aug_full, _, _, X_test_aug = build_features_fold(
    dataAugmented.reset_index(drop=True),
    dataTraining.sample(frac=0.05, random_state=42).reset_index(drop=True),  # val pequeño solo para early stopping
    dataTesting
)

# Sample weights: pseudo-labels pesan PSEUDO_WEIGHT, reales pesan 1.0
sample_weight = np.ones(len(dataAugmented))
sample_weight[len(dataTraining):] = PSEUDO_WEIGHT

m_pseudo = lgb.LGBMRegressor(**params_lgbm)
m_pseudo.fit(X_aug_full, y_aug_full, sample_weight=sample_weight,
             callbacks=[lgb.log_evaluation(period=-1)])

test_pseudo = np.clip(m_pseudo.predict(X_test_aug), 0, 100)

# Blend: 70% mejor ensemble + 30% pseudo-model
test_blended = 0.7 * best_preds + 0.3 * test_pseudo

# OOF del pseudo-model sobre train original para comparar
X_tr_orig, _, _, _, _ = build_features_fold(
    dataTraining, dataTraining.head(10), dataTesting
)
oof_pseudo = np.clip(m_pseudo.predict(X_tr_orig), 0, 100)
rmse_pseudo = np.sqrt(mean_squared_error(y_full, oof_pseudo))
print(f'\n🔁 Pseudo-label model  RMSE sobre train: {rmse_pseudo:.4f}')
print('   (Nota: este RMSE no es comparable directamente — el modelo vio pseudo-labels del test)')
print('\n✅ Blend 70/30 generado. Súbelo junto con el mejor ensemble para comparar en la competencia.')

## 8. Visualización de resultados

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# RMSE por modelo
modelos   = ['LGBM', 'XGB', 'CatBoost', 'HistGBM', 'Simple avg', 'Ponderado', 'Meta XGB']
rmse_vals = [rmse_lgbm_oof, rmse_xgb_oof, rmse_cat_oof, rmse_hgb_oof,
             rmse_simple, rmse_weighted, rmse_stacked]
colores = ['#4C72B0','#DD8452','#55A868','#937860','#8172B2','#8172B2','#C44E52']
bars = axes[0].bar(modelos, rmse_vals, color=colores, alpha=0.85, edgecolor='white')
axes[0].set_ylabel('RMSE OOF')
axes[0].set_title('RMSE por modelo')
axes[0].tick_params(axis='x', rotation=35)
for bar, val in zip(bars, rmse_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=8)

# Scatter real vs predicho
sample_idx = np.random.choice(len(y_full), size=3000, replace=False)
axes[1].scatter(y_full[sample_idx], best_preds[sample_idx], alpha=0.3, s=8, color='#4C72B0')
axes[1].plot([0,100],[0,100], 'r--', linewidth=1, alpha=0.7)
axes[1].set_xlabel('Popularidad real')
axes[1].set_ylabel('Predicción')
axes[1].set_title(f'Real vs Predicho ({best_name})')
axes[1].set_xlim(0,100); axes[1].set_ylim(0,100)

# Distribución de predicciones en test
axes[2].hist(best_preds, bins=50, color='#4C72B0', alpha=0.7, label='Ensemble')
axes[2].hist(test_blended, bins=50, color='#C44E52', alpha=0.5, label='+ Pseudo-label')
axes[2].set_xlabel('Popularidad predicha')
axes[2].set_title('Distribución predicciones test')
axes[2].legend()

plt.tight_layout()
plt.show()

## 9. Feature Importance — ¿qué features mueven el modelo?

In [ ]:
# Reentrenamos LGBM completo para ver importancias
X_full, y_full2, X_vl_tmp, y_vl_tmp, X_test_fi = build_features_fold(
    dataTraining.iloc[:int(len(dataTraining)*0.9)].reset_index(drop=True),
    dataTraining.iloc[int(len(dataTraining)*0.9):].reset_index(drop=True),
    dataTesting
)

m_fi = lgb.LGBMRegressor(**params_lgbm)
m_fi.fit(X_full, y_full2,
         eval_set=[(X_vl_tmp, y_vl_tmp)],
         callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)])

fi = pd.Series(m_fi.feature_importances_, index=X_full.columns).sort_values(ascending=True)
top_fi = fi.tail(30)

fig, ax = plt.subplots(figsize=(10, 8))
top_fi.plot.barh(ax=ax, color='#4C72B0', alpha=0.8)
ax.set_title('Top 30 Features — LightGBM Importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print('\n🔍 Top 15 features más importantes:')
for feat, imp in fi.tail(15).sort_values(ascending=False).items():
    print(f'  {feat:<35} {imp:.0f}')

## 10. Generar submissions

In [ ]:
# Submission del mejor ensemble
sub_best = pd.DataFrame(
    np.clip(best_preds, 0, 100),
    index=dataTesting.index, columns=['Popularity']
)
sub_best.to_csv('test_submission_v6_best.csv', index_label='ID')
print(f'✅ test_submission_v6_best.csv  →  {best_name}  (RMSE OOF: {best_rmse:.4f})')

# Submission con pseudo-labeling blend
sub_blend = pd.DataFrame(
    np.clip(test_blended, 0, 100),
    index=dataTesting.index, columns=['Popularity']
)
sub_blend.to_csv('test_submission_v6_pseudolabel.csv', index_label='ID')
print(f'✅ test_submission_v6_pseudolabel.csv  →  70% ensemble + 30% pseudo-label')

# Todas las variantes
for name, (rmse_val, preds) in scores.items():
    fname = 'test_v6_' + name.lower().replace(' ', '_') + '.csv'
    pd.DataFrame(np.clip(preds, 0, 100),
                 index=dataTesting.index,
                 columns=['Popularity']).to_csv(fname, index_label='ID')

print('\n📊 Estadísticas del mejor ensemble:')
print(sub_best['Popularity'].describe().round(2))

print('\n💡 Estrategia de envíos:')
print('  1. test_submission_v6_best.csv       → ensemble principal')
print('  2. test_submission_v6_pseudolabel.csv → versión con pseudo-labeling')
print('  3. Si tienes submissions sobrantes, prueba las variantes individuales')
print('  4. El pseudo-label puede ganar o perder dependiendo del test — siempre compara')

## 11. Ideas adicionales para seguir subiendo

### A. Más features (impacto alto)
```python
# 1. Artista × Género (interacción cruzada)
artist_genre_mean = train.groupby(['artists','track_genre'])['popularity'].mean()
df['artist_genre_mean'] = df.set_index(['artists','track_genre']).index.map(artist_genre_mean).fillna(global_mean).values

# 2. Rank del artista dentro del género
genre_artist_rank = train.groupby('track_genre')['artist_mean'].rank(pct=True)

# 3. Popularidad cuantílica (en qué percentil está respecto al género)
genre_quantile = train.groupby('track_genre')['popularity'].transform(lambda x: x.rank(pct=True))
```

### B. Más pseudo-labeling (iterativo)
```python
# Hacer 2-3 rondas: cada vez el modelo es mejor y los pseudo-labels más confiables
for round_i in range(3):
    # ... reentrenar con pseudo-labels del round anterior
    PSEUDO_STD_THRESHOLD -= 0.5  # volvemos más restrictivo
```

### C. Búsqueda de hiperparámetros del meta-modelo
```python
# Usar Optuna también para el meta-modelo XGB
study_meta = optuna.create_study(direction='minimize')
study_meta.optimize(objective_meta, n_trials=50)
```

### D. Submission estratégica
- Súbelo todo y compara public score vs OOF
- Si OOF ≈ public score: buen generalization, confía en el OOF
- Si public >> OOF: estás overfitting al val set → aumenta regularización
